<a href="https://colab.research.google.com/github/lilianam37/deep-minds/blob/main/week1/01_whitney_url_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Hook Colab up to Drive and point it at our shared folder
import sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/DeepMinds')
sys.path.append(str(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Load the raw dataset and flip the label so phishing = 1
import pandas as pd
from project_utils import URL_FEATURES

df = pd.read_csv(DATA_DIR / 'PhiUSIIL_Phishing_URL_Dataset.csv')
df['label'] = 1 - df['label']
print(df.shape)
print(df['label'].value_counts())

missing = [c for c in URL_FEATURES if c not in df.columns]
assert not missing, f'Feature names not found: {missing}'

(235795, 56)
label
0    134850
1    100945
Name: count, dtype: int64


In [ ]:
#Split the rows 70/15/15 with the same settings as before, keeping the URL text this time
from sklearn.model_selection import train_test_split

idx_train, idx_temp = train_test_split(df.index, test_size=0.30, stratify=df['label'], random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=df.loc[idx_temp, 'label'], random_state=42)

df['split'] = 'train'
df.loc[idx_val, 'split'] = 'val'
df.loc[idx_test, 'split'] = 'test'
print(df['split'].value_counts())

split
train    165056
test      35370
val       35369
Name: count, dtype: int64


In [ ]:
#Must match the midterm split exactly
for name, idx in [('train', idx_train), ('val', idx_val), ('test', idx_test)]:
    old = pd.read_csv(DATA_DIR / f'y_{name}.csv')['label'].to_numpy()
    new = df.loc[idx, 'label'].to_numpy()
    print(name, 'matches old split:', len(old) == len(new) and (old == new).all())

train matches old split: True
val matches old split: True
test matches old split: True


In [ ]:
#2nd split for a later experiment (no website domain appears in more than one split)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
g_train, g_temp = next(gss.split(df, groups=df['Domain']))
temp = df.iloc[g_temp]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
g_val, g_test = next(gss2.split(temp, groups=temp['Domain']))

df['split_domain'] = 'train'
df.loc[temp.index[g_val], 'split_domain'] = 'val'
df.loc[temp.index[g_test], 'split_domain'] = 'test'

print(df.groupby('split_domain')['label'].agg(['count', 'mean']).rename(columns={'mean': 'phishing share'}))
tr = set(df.loc[df.split_domain == 'train', 'Domain'])
print('Domains shared between train and test:', len(tr & set(df.loc[df.split_domain == 'test', 'Domain'])))
print('Domains shared in the RANDOM split:', len(set(df.loc[df.split == 'train', 'Domain']) & set(df.loc[df.split == 'test', 'Domain'])))

               count  phishing share
split_domain                        
test           34963        0.424020
train         165452        0.429285
val            35380        0.426625
Domains shared between train and test: 0
Domains shared in the RANDOM split: 1543


In [ ]:
#Saving one master file that everyone else's notebooks read from
keep = ['URL', 'Domain', 'TLD', 'label', 'split', 'split_domain'] + URL_FEATURES
df[keep].to_csv(DATA_DIR / 'master_with_splits.csv', index=False)
print('Saved', DATA_DIR / 'master_with_splits.csv', df[keep].shape)

Saved /content/drive/MyDrive/DeepMinds/master_with_splits.csv (235795, 28)


1,543 domains shared in the random split.